# Full Pipeline

Every chapter so far has taken one step of the reduction, looked at what it did, and chosen its
settings by inspection. This chapter takes none of those decisions. It composes the calls the
book has taught into a single reduction that runs without anyone looking, from the raw frames to
a wavelength-calibrated spectrum on disk, and every setting in it is the one an earlier chapter
arrived at, cited where it is used.

The reduction has three stages, and they are the book's parts in order: the arcs give a tilt
solution, the rectified arcs give a wavelength solution, and the science frame goes through
tracing, background subtraction, extraction and calibration. Each stage is one function, so that
a stage can be read, replaced, or dropped on its own.

The tilt stage is the one most likely to be dropped. It is here because the OSIRIS frames need
it; on an instrument whose lines are already straight, the pipeline is the same code with the
two `resample` calls removed and the frames used as they come.

In [ ]:
import sys
import warnings

sys.path.append("../src")

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.modeling import models
from specreduce.background import Background
from specreduce.extract import BoxcarExtract, HorneExtract
from specreduce.tilt_correction import TiltCorrection
from specreduce.tracing import FitTrace, FlatTrace
from specreduce.wavecal1d import WavelengthCalibration1D
from specreduce.wavesol1d import WavelengthSolution1D
from specreduce.wavesol2d import WavelengthSolution2D

from common import (
    DATA_DIR,
    GTC_HGAR,
    WAVELENGTH_SOLUTION_FILE,
    fit_wavelength_solution,
    read_data,
)

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()

## Configuration

A pipeline carries its instrument knowledge as a handful of numbers. Here they are the reference
row on the slit, chosen in Chapter 4 and used by every calibration since, and the prior on the
wavelength solution that Chapter 11 introduced: the wavelength at the reference pixel, the
dispersion there, and the two curvature terms, taken from the interactive calibration of
Chapter 10. In a real pipeline this block is a configuration entry per instrument setup, and the
values come from a set of calibrations that somebody has checked.

In [ ]:
REFERENCE_ROW = 135

PRIOR = dict(
    wavelength=7521.6,  # A, at the reference pixel 512
    dispersion=5.291,  # A / pix, at the reference pixel
    quadratic=8.77e-4,
    cubic=-3.96e-7,
)

## Stage one: the tilt solution

Chapter 6, unchanged: seven cross-dispersion samples, lines found at `noise_factor=25`, a
fourth-degree surface, and one refinement at a one-pixel match bound.

In [ ]:
def calibrate_tilt(arcs):
    """Fit the tilt of the arc lines and return the solution that straightens them."""
    tilt = TiltCorrection(arcs, FlatTrace(arcs[0], REFERENCE_ROW), n_cdisp_samples=7)
    tilt.find_arc_lines(fwhm=2.5, noise_factor=25)
    tilt.fit(degree=4)
    tilt.refine_fit(degree=4, match_distance_bound=1.0)
    return tilt.solution


tilt = calibrate_tilt(arcs)

## Stage two: the wavelength solution

Chapter 11's automated workflow. The arcs are rectified and extracted at the reference row,
`find_lines` detects the lines with Chapter 9's settings, `fit_dispersion` searches inside bounds
drawn around the prior, and one `refine_fit` at 2 Å settles the matched set. The seed makes the
search reproducible.

In [ ]:
def calibrate_wavelength(arcs, tilt, prior, seed=0):
    """Fit the 1D wavelength solution automatically from a prior on its coefficients."""
    reference = FlatTrace(arcs[0], REFERENCE_ROW)
    spectra = [
        BoxcarExtract(tilt.resample(frame), reference, width=15).spectrum
        for frame in arcs
    ]
    calibration = WavelengthCalibration1D(
        arc_spectra=spectra,
        line_lists=[GTC_HGAR, ["NeI"], ["XeI"]],
        line_list_bounds=(5000, 10500),
        ref_pixel=512,
        unit=u.angstrom,
        wave_air=True,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # unconverged refits on non-line detections
        calibration.find_lines(fwhm=2.5, noise_factor=15)
        calibration.fit_dispersion(
            wavelength_bounds=(prior["wavelength"] - 150, prior["wavelength"] + 150),
            dispersion_bounds=(prior["dispersion"] - 0.2, prior["dispersion"] + 0.2),
            higher_order_limits=[3 * abs(prior["quadratic"]), 3 * abs(prior["cubic"])],
            degree=3,
            seed=seed,
        )
        calibration.refine_fit(max_match_distance=2.0)
    calibration.remove_unmatched_lines()
    return calibration


calibration = calibrate_wavelength(arcs, tilt, PRIOR)
wavelength = calibration.solution

matched = sum(len(lines) for lines in calibration.observed_lines)
print(f"rms {calibration.rms('wavelength'):.3f} A over {matched} lines")

That is the one place the pipeline reports on itself, and the numbers are the ones a convergence
check would read, as Chapter 11 discussed. Here the solution can also be compared with the
interactive one Chapter 10 stored, which a pipeline cannot do:

In [ ]:
if not WAVELENGTH_SOLUTION_FILE.exists():  # normally written by Chapter 10
    fit_wavelength_solution(arcs, tilt).to_asdf(WAVELENGTH_SOLUTION_FILE)
interactive = WavelengthSolution1D.from_asdf(WAVELENGTH_SOLUTION_FILE)

pixels = np.arange(1000.0)
deviation = np.max(
    np.abs(wavelength.pix_to_wav(pixels) - interactive.pix_to_wav(pixels))
)
print(f"largest difference from Chapter 10's solution: {deviation:.2f} A")

## Stage three: the science frame

Chapters 4, 7, 8 and 10 in order. The frame is rectified, the trace is fitted with 32 bins and a
fourth-degree Chebyshev polynomial, the background is measured in two windows 60 rows from the
trace and 20 rows wide, the spectrum is extracted with `HorneExtract`, and the result is put on a
wavelength grid with `resample`. Extraction happens on the rectified frame, the order the book has
used throughout; Chapter 12 covers the alternative of calibrating the 2D frame first, and the
case for extracting on the raw detector with a routine of your own.

In [ ]:
def reduce(frame, tilt, wavelength):
    """Take one science frame from the detector to a calibrated 1D spectrum."""
    rectified = tilt.resample(frame)
    trace = FitTrace(
        rectified, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
    )
    subtracted = Background.two_sided(
        rectified, trace, separation=60, width=20
    ).sub_image()
    extracted = HorneExtract(subtracted, trace).spectrum
    return wavelength.resample(extracted), extracted


spectrum, extracted = reduce(obj, tilt, wavelength)

fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
ax.plot(spectrum.spectral_axis.value, spectrum.flux.value, lw=0.6)
ax.set(
    xlim=(spectrum.spectral_axis.value[0], spectrum.spectral_axis.value[-1]),
    xlabel=r"Wavelength [$\AA$]",
    ylabel=f"Flux [{spectrum.flux.unit.to_string('latex_inline')}]",
);

## The products

Four files leave the pipeline. The tilt solution, the wavelength solution and their combination
as a `WavelengthSolution2D` are calibration products, one set per instrument configuration, and
they are written as ASDF the way Chapters 6, 10 and 12 wrote them. The spectrum is written to
FITS through `attach_wcs`, which leaves the samples on the extraction's pixel grid and records
the wavelength solution as a standard spectral WCS in the header, so that any FITS reader can
put the axis back. `resample` is applied for the plot above; the file keeps the unresampled
data.

In [ ]:
products = DATA_DIR / "pipeline"
products.mkdir(exist_ok=True)

tilt.to_asdf(products / "tilt_solution.asdf")
wavelength.to_asdf(products / "wavelength_solution.asdf")
WavelengthSolution2D(tilt, wavelength).to_asdf(products / "wavelength_solution_2d.asdf")
wavelength.attach_wcs(extracted).write(
    products / "tres3b_spectrum.fits", format="wcs1d-fits", overwrite=True
)

print("\n".join(path.name for path in sorted(products.iterdir())))

## Summary

- The whole reduction is **three functions in the book's order**: arcs to a tilt solution, rectified
  arcs to a wavelength solution, science frame to a calibrated spectrum. Nothing is inspected;
  every setting is one an earlier chapter chose and explained.
- The pipeline's instrument knowledge is a **configuration block**: the reference row, and the
  prior on the wavelength solution that the automated `fit_dispersion` searches around.
- The tilt stage is **instrument-specific**. Without it, the pipeline is the same code with the two
  `resample` calls removed.
- The products are the three solutions as **ASDF** and the spectrum as **FITS with a spectral WCS**,
  written through `attach_wcs` so the data stay on their pixel grid.
- The automated wavelength solution agrees with the interactive one of Chapter 10 to a fraction of
  a pixel, which is the check the book can make and a pipeline cannot; a pipeline reads the rms
  and the matched line count instead.